# 04 - Sea ice and waves: floe-size distribution diagnostics

Minimal diagnostics for the WW3 -> CICE floe-size-distribution (FSD) coupling in a MOM6 + CICE + WW3 regional run: five maps and a movie. Point `CASE` at any run with the CICE FSD fields and the mediator wave stream and re-run.

Adapted from `bering_wave_fsd.ipynb` (Dan Amrhein and Claude). The heavier analysis there (process budgets, closure checks, spectral transmission) lives in `bering_fsd.py`.

## Setup

History files are opened directly in the cells below; only the scientific operations are wrapped in functions.

In [ ]:
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}
import glob
import os
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean.cm as cmo
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap, LogNorm, Normalize
from IPython.display import Image

from mom6_tools.jobqueue import get_cluster

xr.set_options(keep_attrs=True)

FIGDIR = Path("diagnostic_figures")
FIGDIR.mkdir(exist_ok=True)

### Start a Dask Cluster/Client
Dask is a parallel computation library for Python that lets us initialize multiple workers to act as parallel processors. There are two options for creating a dask cluster:
- PBSCluster: This submits new PBS jobs for each worker, running separate jobs on Casper/Derecho for each worker.
- LocalCluster: If you are running this notebook on an existing PBS job with enough resources, it can access those resources directly.

In [ ]:
DASK = "PBSCluster" # or LocalCluster (works with SLURMCluster on other systems)

N_WORKERS = 4

ACCOUNT = "P93300012" # charge worker jobs here, defaults to PROJECT and PBS_ACCOUNT env variables in bash
QUEUE = "casper" # or "derecho"
INTERFACE = 'ext' # or "ib0" for derecho

# Following are resources for each worker
# Total resources are N_WORKERS * resource (e.g. N_WORKERS * MEMORY).
MEMORY = "20GB"
CORES = 4
PROCESSES = 1
WALLTIME = "03:00:00"

success, client, cluster = get_cluster(nw=N_WORKERS, cluster_class=DASK, account=ACCOUNT, queue=QUEUE, walltime=WALLTIME, memory=MEMORY, processes=PROCESSES, cores = CORES, interface=INTERFACE, log_directory="logs",)

In [ ]:
client

## Case Selection

In [ ]:
# --- Case configuration ---
CASE = dict(
        casename="bering_ocn_wav_ice.001",
        archive="/glade/derecho/scratch/ajanney/archive/bering_ocn_wav_ice.001",
        date_range=["2020-03-01", "2020-03-31"],
        # CICE history stream holding the FSD fields, and the suffix CICE appends to
        # every field name in it: ("h", "") for a lone monthly stream,
        # ("h1", "_d") for a daily stream (histfreq = "m","d",... in user_nl_cice).
        ice_stream="h",
        ice_suffix="",
    )

ARCHIVE = CASE["archive"]
DATE_RANGE = CASE["date_range"]
FILE_PATTERN = f"{ARCHIVE}/ocn/hist/{CASE['casename']}.mom6"

# WW3 writes no gridded history unless HIST_OPTION/HIST_N are set, so the wave
# fields come from the CMEPS mediator stream in cpl/hist.
ICE_FILES = sorted(glob.glob(f"{ARCHIVE}/ice/hist/{CASE['casename']}.cice.{CASE['ice_stream']}.*.nc"))
WAV_FILES = sorted(glob.glob(f"{ARCHIVE}/cpl/hist/{CASE['casename']}.cpl.hx.ww3.*.nc"))

print(f"{CASE['casename']}: {len(ICE_FILES)} CICE '{CASE['ice_stream']}' file(s), "
      f"{len(WAV_FILES)} mediator wave file(s)")

### Relevant Files
- STATIC - `casename.mom6.h.static.nc`: contains real lat/lon coords, grid information, coriolis parameter
- GEOM - `casename.mom6.h.ocean_geometry.nc`: a lot of similar info to STATIC, includes bathymetry
- ICE - `casename.cice.h*.nc`: CICE history, including the FSD fields `afsd`, `fsdrad` and the tendencies `dafsd_*`
- WAV - `casename.cpl.hx.ww3.*.nc`: WW3 fields as the mediator received them (`wavImp_Sw_*`)

CICE, WW3 and MOM6 share the tracer grid in these regional runs, so the ice and wave streams are renamed onto the MOM6 dimensions (`yh`, `xh`) and plot with the same `MAP["h"]` keywords as any ocean tracer.

In [ ]:
# --- Grid files ---
# The static file carries the land masks, cell areas and Coriolis parameter.
STATIC = xr.open_dataset(f"{FILE_PATTERN}.h.static.nc", decode_times=False).squeeze(drop=True)

# Rename geometry coords to the history-file names.
GEOM = (xr.open_dataset(f"{FILE_PATTERN}.h.ocean_geometry.nc")
          .rename_dims({"lath": "yh", "lonh": "xh", "latq": "yq", "lonq": "xq"})
          .drop_vars(["lath", "lonh", "latq", "lonq"]))

# Attach these to any history DATASET with `ds = ds.assign_coords(COORDS)`
# This streamlines plotting and xgcm grid operations, which expect the geometry coords to be present.
COORDS = {name: GEOM[name] for name in
          ("geolon", "geolat", "geolonu", "geolatu",
           "geolonv", "geolatv", "geolonb", "geolatb")}

# Example: STATIC now stores geolon/geolat as data variables (psst it already had them)
STATIC = STATIC.assign_coords(COORDS)

print(dict(STATIC.sizes))
print("lon {:.2f} .. {:.2f}   lat {:.2f} .. {:.2f}".format(
    float(GEOM.geolon.min()), float(GEOM.geolon.max()),
    float(GEOM.geolat.min()), float(GEOM.geolat.max())))

### Formatting Plots
For efficiency in these notebooks, we provide some standard plot settings and information. This lets us quickly use the implicit plotting capapbilities of xarray (i.e. dataarray.plot(kwargs**)).

In [ ]:
# --- Standard plotting keywords ---
# Everything is plotted with xarray's own `.plot()`; these dicts just supply the
# coordinates and the transform so everything is standardized and pretty!

SUBPLOT = {"projection": ccrs.Robinson(central_longitude=float(GEOM.geolon.mean()))}
LAND = cfeature.LAND.with_scale("50m")

_aspect = float(GEOM.geolat.max() - GEOM.geolat.min()) / float(GEOM.geolon.max() - GEOM.geolon.min())
PANEL = np.array([18.0, round(18.0 * _aspect)])

# Lat lon gridlines for all plots. Use `ax.gridlines(**GRID)` to apply.
nticks = 7
GRID = dict(
      draw_labels=["bottom", "left"],
      xlocs=mticker.MultipleLocator(np.round(np.abs((GEOM.geolon.max().values - GEOM.geolon.min().values) / nticks), 1)),
      ylocs=mticker.MultipleLocator(np.round(np.abs((GEOM.geolat.max().values - GEOM.geolat.min().values) / nticks), 1)),
      linewidth=0.5, color="0.5", alpha=0.5,
  )

# Fast map for plotting MOM6 fields. Use `ds.plot(**MAP[grid])` where grid is one of "h", "u", "v", or "q".
# h - tracers, q - corners, u - u-velocity, v - v-velocity
MAP = {
    "h": dict(x="geolon",  y="geolat",  transform=ccrs.PlateCarree()),
    "u": dict(x="geolonu", y="geolatu", transform=ccrs.PlateCarree()),
    "v": dict(x="geolonv", y="geolatv", transform=ccrs.PlateCarree()),
    "q": dict(x="geolonb", y="geolatb", transform=ccrs.PlateCarree()),
}

# add-ons to spread over a `.plot()` call
DIFF = dict(cmap="cmo.balance", center=0, robust=True)   # diverging / difference
SECTION = dict(yincrease=False)                          # depth down the y axis

In [ ]:
# --- Ice edge overlay ---
# Draw the ice edge on any map with `aice.plot.contour(ax=ax, **MAP["h"], **EDGE)`.
ICE_EDGE = 0.15      # conventional ice edge (aice)
EDGE = dict(levels=[ICE_EDGE], colors="w", linewidths=1.4)

### Ice, wave and GIF helpers
The FSD bin edges are not written to the CICE history files, so they are hard-coded from `icepack_fsd.F90` (nfsd = 12). `afsd` is a density per metre of floe radius, but every `dafsd_*` tendency is a plain per-bin rate: it is summed over bins, never multiplied by the bin width.

The mediator wave records are period means stamped at the *end* of the period; `open_ww3_mediator` moves each stamp to the middle, and `match_to_ice` averages the waves onto each ice record's time bounds so that ice and wave fields share one time axis.

In [ ]:
# --- CICE / floe-size distribution ---
FSD_LIMS = np.array([
    6.65000000e-02, 5.31030847e+00, 1.42865861e+01, 2.90576686e+01,
    5.24122136e+01, 8.78691405e+01, 1.39518470e+02, 2.11635752e+02,
    3.08037274e+02, 4.31203059e+02, 5.81277225e+02, 7.55141047e+02,
    9.45812834e+02])
FLOE_RAD_C = 0.5 * (FSD_LIMS[:-1] + FSD_LIMS[1:])   # bin centres, m
FLOE_BINWIDTH = np.diff(FSD_LIMS)                   # dr, m

SMALL_FLOE = 29.0    # m -- the three smallest bins; the break-up end of the spectrum


def open_cice(paths, suffix="", chunks={"time": 1}):
    """Open CICE history files on the MOM6 tracer-grid dimension names.

    CICE and MOM6 share the tracer grid in these regional cases, so nj/ni are
    renamed to yh/xh and the notebook's COORDS / MAP["h"] apply unchanged.
    `suffix` is the stream suffix CICE appends to every field ("_d" for a daily
    h1 stream, "" for a lone monthly stream); it is stripped so the notebook
    can use plain names like "aice".
    """
    ds = xr.open_mfdataset(paths, combine="by_coords", chunks=chunks,
                           decode_timedelta=False)
    if suffix:
        ds = ds.rename({v: v[:-len(suffix)] for v in ds.data_vars if v.endswith(suffix)})
    return ds.rename({"nj": "yh", "ni": "xh"})


def small_floe_fraction(ice):
    """Area fraction in floes smaller than SMALL_FLOE, masked to ice-covered cells."""
    k = np.where(FLOE_RAD_C < SMALL_FLOE)[0]
    dr = xr.DataArray(FLOE_BINWIDTH[k], dims="nf")
    return (ice["afsd"].isel(nf=k) * dr).sum("nf").where(ice["aice"] > ICE_EDGE)


def fracturing_rate(ice):
    """Positive part of the wave FSD tendency, summed over bins [s-1].

    Not weighted by bin width: dafsd_wave is already a per-bin rate (see the
    module notes), and weighting by dr breaks its area conservation.
    """
    d = ice["dafsd_wave"]
    return d.where(d > 0, 0.0).sum("nf")

In [ ]:
# --- WW3 mediator stream ---
WAVE_MAX = 100.0     # m -- guard for the CICE wave_sig_ht artifact


def open_ww3_mediator(paths, chunks={"time": 1}):
    """Open the CMEPS mediator wave stream on the MOM6 tracer-grid dimension names,
    with each record re-stamped at the middle of its averaging period."""
    wav = xr.open_mfdataset(paths, combine="by_coords", chunks=chunks,
                            decode_timedelta=False)
    wav = wav.rename({"wavImp_ny": "yh", "wavImp_nx": "xh"})
    dt = wav["time"].diff("time").median().values
    return wav.assign_coords(time=wav["time"].values - dt / 2)


def match_to_ice(wav, ice):
    """Average the wave records onto each ice record's time_bounds window.

    Daily waves against monthly ice become monthly means; against daily ice
    each window holds one record. Returned on the ice time axis, so ice*wave
    expressions align.
    """
    bnds = ice["time_bounds"].values
    out = [wav.sel(time=slice(t0, t1)).mean("time") for t0, t1 in bnds]
    return xr.concat(out, dim="time").assign_coords(time=ice["time"].values)


def significant_wave_height(ice, wav):
    """Significant wave height [m]: the mediator's wavImp_Sw_hs_avg when present,
    otherwise CICE's in-ice wave_sig_ht with the artifact masked."""
    if wav is not None and "wavImp_Sw_hs_avg" in wav:
        return wav["wavImp_Sw_hs_avg"]
    h = ice["wave_sig_ht"]
    return h.where(h < WAVE_MAX)

In [ ]:
# --- GIF helpers ---
def save_gif(frame_paths, out_path, duration=350, loop=0, clean = False):
    """Stitch saved PNG frames into an animated GIF with Pillow.
    """
    from PIL import Image

    frames = [Image.open(p).convert("P", palette=Image.ADAPTIVE)
              for p in frame_paths]
    out_path = Path(out_path)
    frames[0].save(out_path, save_all=True, append_images=frames[1:],
                   duration=duration, loop=loop, optimize=True)

    if clean:
        for p in frame_paths:
            os.remove(p)

    return out_path


def save_field_gif(field, frames_dir, out_path, subplot_kw, map_kw, figsize, land=None,
                   gridlines_kw=None, cmap="cmo.thermal", coastlines="10m",
                   title=None, dpi=90, duration=100, loop=0, clean=False):
    """One PNG per time record of `field`, stitched into an animated GIF.

    This is a little clunkier than keeping everything in Jupyter or Python memory, but saving
    out PNGs and then assembling into a GIF is much more efficient. Frames are written to
    `frames_dir`, and `clean` removes them once stitched.
    """
    out_path = Path(out_path)
    frames_dir = Path(frames_dir)
    frames_dir.mkdir(parents=True, exist_ok=True)
    vmin, vmax = float(field.min()), float(field.max())
    label = field.name if title is None else title

    frames = []
    for i in range(field.sizes["time"]):
        fig, ax = plt.subplots(figsize=figsize, subplot_kw=subplot_kw)
        field.isel(time=i).plot(ax=ax, **map_kw, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.coastlines(coastlines)
        if land is not None:
            ax.add_feature(land, facecolor="0.85")
        if gridlines_kw is not None:
            ax.gridlines(**gridlines_kw)
        fig.suptitle(f"{label}   {str(field.time.values[i])[:10]}")
        p = frames_dir / f"{out_path.stem}_frame_{i:03d}.png"
        fig.savefig(p, dpi=dpi)
        plt.close(fig)
        frames.append(p)

    return save_gif(frames, out_path, duration=duration, loop=loop, clean=clean)

## 1. Opening the ice and wave streams

The static maps are drawn for the ice record with the largest ice area, where the pack and the waves coexist.

In [ ]:
# from mom6_tools.regional_cicetoolbox import open_cice
# from mom6_tools.regional_ww3toolbox import open_ww3_mediator, match_to_ice, significant_wave_height

ice = open_cice(ICE_FILES, suffix=CASE["ice_suffix"]).sel(time=slice(*DATE_RANGE))
ice = ice.assign_coords(geolon=GEOM.geolon, geolat=GEOM.geolat)

wav_raw = open_ww3_mediator(WAV_FILES).assign_coords(geolon=GEOM.geolon, geolat=GEOM.geolat) if WAV_FILES else None
wav = match_to_ice(wav_raw, ice) if WAV_FILES else None
hs = significant_wave_height(ice, wav).where(STATIC.wet)

DATES = [str(t)[:10] for t in ice.time.values]
area = (ice["aice"] * ice["tarea"]).sum(("yh", "xh")).compute()
DAY = int(area.argmax("time"))

print(f"ice: {len(DATES)} record(s), {DATES[0]} .. {DATES[-1]}")
if wav_raw is not None:
    print(f"waves: {wav_raw.sizes['time']} mediator record(s), averaged onto the ice records")
print(f"maps drawn for the ice-area maximum: {DATES[DAY]}")

## 2. The forcing: significant wave height and where the ice is

White contour is the 0.15 ice edge. This is the geometry every other map refers to.

In [ ]:
# from mom6_tools.regional_ww3toolbox import significant_wave_height

fig, ax = plt.subplots(figsize=PANEL, subplot_kw=SUBPLOT)
hs.isel(time=DAY).plot(ax=ax, **MAP["h"], cmap="magma", vmin=0, robust=True,
                       cbar_kwargs={"label": "$H_s$ [m]"})

ice["aice"].isel(time=DAY).plot.contour(ax=ax, **MAP["h"], **EDGE)
ax.coastlines("50m")
ax.add_feature(LAND, facecolor="0.85")
gl = ax.gridlines(**GRID)

fig.suptitle(f"{CASE['casename']}: significant wave height — {DATES[DAY]}")
fig.savefig(FIGDIR / "04_hs.png", dpi=150)
plt.show()

## 3. The FSD state: mean floe radius

Masked to ice-covered cells. Large values are a coarse, unbroken pack.

In [ ]:
fsdrad = ice["fsdrad"].isel(time=DAY).where(ice["aice"].isel(time=DAY) > ICE_EDGE)

fig, ax = plt.subplots(figsize=PANEL, subplot_kw=SUBPLOT)
fsdrad.plot(ax=ax, **MAP["h"], cmap="cividis", vmin=0, robust=True,
            cbar_kwargs={"label": "fsdrad [m]"})

ice["aice"].isel(time=DAY).plot.contour(ax=ax, **MAP["h"], **EDGE)
ax.coastlines("50m")
ax.add_feature(LAND, facecolor="0.85")
gl = ax.gridlines(**GRID)

fig.suptitle(f"{CASE['casename']}: mean floe radius — {DATES[DAY]}")
fig.savefig(FIGDIR / "04_fsdrad.png", dpi=150)
plt.show()

## 4. Where the ice is broken: small-floe area fraction

Area fraction in floes smaller than 29 m — the three smallest FSD bins, i.e. the break-up end of the spectrum.

In [ ]:
# from mom6_tools.regional_cicetoolbox import small_floe_fraction

fig, ax = plt.subplots(figsize=PANEL, subplot_kw=SUBPLOT)
small_floe_fraction(ice).isel(time=DAY).plot(ax=ax, **MAP["h"], cmap="YlOrRd", vmin=0, robust=True,
                                             cbar_kwargs={"label": "area fraction"})

ice["aice"].isel(time=DAY).plot.contour(ax=ax, **MAP["h"], **EDGE)
ax.coastlines("50m")
ax.add_feature(LAND, facecolor="0.85")
gl = ax.gridlines(**GRID)

fig.suptitle(f"{CASE['casename']}: small-floe area fraction ($r<29$ m) — {DATES[DAY]}")
fig.savefig(FIGDIR / "04_small_floe_fraction.png", dpi=150)
plt.show()

## 5. The coupling term firing: wave fracturing rate

Positive part of `dafsd_wave` summed over floe-size bins. Log scale over five decades below the maximum; zeros are left transparent, so blank means genuinely no fracturing, not a small value. The magnitude depends on the history averaging period (monthly means are far smaller than daily peaks), so the colour range follows the data.

In [ ]:
# from mom6_tools.regional_cicetoolbox import fracturing_rate

fr = fracturing_rate(ice).isel(time=DAY).where(STATIC.wet).compute()
fr_max = float(fr.max())

fig, ax = plt.subplots(figsize=PANEL, subplot_kw=SUBPLOT)
fr.where(fr > 0).plot(ax=ax, **MAP["h"], cmap="inferno", norm=LogNorm(vmin=fr_max * 1e-5, vmax=fr_max),
                      cbar_kwargs={"label": "fracturing rate [s$^{-1}$]"})

ice["aice"].isel(time=DAY).plot.contour(ax=ax, **MAP["h"], **EDGE)
ax.coastlines("50m")
ax.add_feature(LAND, facecolor="0.85")
gl = ax.gridlines(**GRID)

fig.suptitle(f"{CASE['casename']}: wave fracturing rate — {DATES[DAY]}")
fig.savefig(FIGDIR / "04_fracturing_rate.png", dpi=150)
plt.show()

## 6. The footprint: where waves ever break ice

Maximum fracturing rate over every ice record. Map 5 is one record and can be mostly empty because the coupling is intermittent; this shows the cumulative reach of the mechanism. No ice edge is drawn, since it moves over the record.

In [ ]:
# from mom6_tools.regional_cicetoolbox import fracturing_rate

frmax = fracturing_rate(ice).max("time").where(STATIC.wet).compute()
frmax_max = float(frmax.max())

fig, ax = plt.subplots(figsize=PANEL, subplot_kw=SUBPLOT)
frmax.where(frmax > 0).plot(ax=ax, **MAP["h"], cmap="inferno",
                            norm=LogNorm(vmin=frmax_max * 1e-5, vmax=frmax_max),
                            cbar_kwargs={"label": "fracturing rate [s$^{-1}$]"})
ax.coastlines("50m")
ax.add_feature(LAND, facecolor="0.85")
gl = ax.gridlines(**GRID)
fig.suptitle(f"{CASE['casename']}: maximum wave fracturing rate, {DATES[0]} .. {DATES[-1]}")
fig.savefig(FIGDIR / "04_fracturing_rate_max.png", dpi=150)
plt.show()

## 7. The movie: waves at the ice edge

One frame per ice record. Open water is coloured by **wave height** (warm) and ice by **mean floe radius** (cool, bright = large floes), with the 0.15 ice edge in cyan. Hue does the separating, so ice never reads as ocean. The strip below carries ice area and the domain-integrated fracturing flux, with a cursor on the current record.

**This demo cases has monthly output, so this cell doesn't run by default.**

In [ ]:
# # from mom6_tools.regional_cicetoolbox import fracturing_rate
# # from mom6_tools.regional_m6toolbox import save_gif

# SEA = LinearSegmentedColormap.from_list(
#     "sea", ["#07070f", "#3d1040", "#8f1d45", "#d94f28", "#f6a623", "#ffe9a8"])
# FLOE = LinearSegmentedColormap.from_list(
#     "floe", ["#1d3350", "#3f6d99", "#86b2d4", "#cfe4f3", "#ffffff"])

# aice = ice["aice"].where(STATIC.wet).load()
# fsd = ice["fsdrad"].load()
# HS = hs.load()
# flux = (fracturing_rate(ice) * ice["aice"] * ice["tarea"]).sum(("yh", "xh")).values

# # one colour range for every frame
# sea_norm = Normalize(0, float(HS.quantile(0.98)))
# floe_norm = Normalize(0, float(fsd.where(aice >= ICE_EDGE).quantile(0.98)))
# x = np.arange(len(DATES))
# tick = max(1, len(x) // 10)

# frames = []
# for k in range(len(DATES)):
#     fig = plt.figure(figsize=(PANEL[0], PANEL[1] + 2.5))
#     gs = fig.add_gridspec(2, 1, height_ratios=[PANEL[1], 2.0])
#     axm = fig.add_subplot(gs[0], **SUBPLOT)
#     axt = fig.add_subplot(gs[1])

#     HS.isel(time=k).where(aice.isel(time=k) < ICE_EDGE).plot(
#         ax=axm, **MAP["h"], cmap=SEA, norm=sea_norm, cbar_kwargs={"label": "$H_s$ [m]", "shrink": 0.8})
#     fsd.isel(time=k).where(aice.isel(time=k) >= ICE_EDGE).plot(
#         ax=axm, **MAP["h"], cmap=FLOE, norm=floe_norm, cbar_kwargs={"label": "mean floe radius [m]", "shrink": 0.8})
#     aice.isel(time=k).plot.contour(ax=axm, **MAP["h"], levels=[ICE_EDGE], colors="#7dfcff", linewidths=1.2)
#     axm.coastlines("50m")
#     axm.add_feature(LAND, facecolor="0.85")
#     gl = axm.gridlines(**GRID)

#     axt.plot(x, area / area.max(), color="#3f6d99", lw=1.3, marker="o", ms=3, label="ice area")
#     axt.plot(x, flux / max(flux.max(), 1e-30), color="#d94f28", lw=1.1, marker="o", ms=3,
#              label="wave fracturing flux")
#     axt.axvline(k, color="k", lw=1.1)
#     axt.set_ylim(-0.05, 1.12)
#     axt.set_xlim(-0.5, len(x) - 0.5)
#     axt.set_xticks(x[::tick])
#     axt.set_xticklabels([DATES[i] for i in x[::tick]], fontsize=8)
#     axt.set_yticks([])
#     axt.set_ylabel("normalised")
#     axt.legend(fontsize=8, ncol=2, loc="upper right")

#     fig.suptitle(f"{CASE['casename']}: waves and floe size — {DATES[k]}")
#     p = FIGDIR / f"04_wave_fsd_frame_{k:03d}.png"
#     fig.savefig(p, dpi=90)
#     plt.close(fig)
#     frames.append(p)

# gif = save_gif(frames, FIGDIR / "04_wave_fsd.gif", duration=250, clean=True)
# print(f"{gif}  ({len(frames)} frame(s))")
# Image(str(gif))

## 8. Daily wave height from the mediator

The mediator stream is written every coupling-history period regardless of the CICE history frequency, so the waves can be animated at their own cadence even when the ice stream is monthly. Each record is a period mean.

In [ ]:
# from mom6_tools.regional_m6toolbox import save_field_gif

field = wav_raw["wavImp_Sw_hs_avg"].sel(time=slice(*DATE_RANGE)).where(STATIC.wet).load()

gif = save_field_gif(field, FIGDIR / "frames", FIGDIR / "04_hs_daily.gif", SUBPLOT, MAP["h"], PANEL,
                     land=LAND, gridlines_kw=GRID, cmap="magma", coastlines="50m",
                     title="significant wave height [m]", duration=150, clean=True)
print(f"{gif}  ({field.sizes['time']} frames)")
Image(str(gif))

## Cleanup

In [ ]:
done = False # let's you use Run All without closing the cluster.
if done:
    client.close()
    cluster.close()